In [ ]:
# Connection Test: PostgreSQL + Doris + DB2i
# # 🔗 Connection Test: PostgreSQL + Doris + DB2i
# Tests connectivity to all data sources using environment variables from mounted .env
# ✨ Features: Optional table check + Username logging in output

# %%
import os
import sys
from pathlib import Path
from pyspark.sql import SparkSession
from pyspark.sql.functions import lit, current_timestamp
import socket

# ============================================
# 🔧 CONFIGURATION: Optional Table Names
# Set to None or "" to skip table-specific checks (connection-only mode)
# ============================================
POSTGRES_TEST_TABLE = os.getenv("POSTGRES_TEST_TABLE", "").strip() or None
DORIS_TEST_TABLE = os.getenv("DORIS_TEST_TABLE", "").strip() or None
DB2I_TEST_TABLE = os.getenv("DB2I_TEST_TABLE", "").strip() or None

print(f"🔧 Config: POSTGRES_TEST_TABLE={POSTGRES_TEST_TABLE}, DORIS_TEST_TABLE={DORIS_TEST_TABLE}, DB2I_TEST_TABLE={DB2I_TEST_TABLE}")

# ============================================
# Step 1: Load Environment Variables from Mounted .env
# ============================================
def load_env_file(env_path="/home/jovyan/.env"):
    """Load .env file into os.environ with fallback paths"""
    env_paths = [
        Path(env_path),
        Path("/home/jovyan/.env"),
        Path(".env"),
        Path("/home/jovyan/work/.env")
    ]
    
    env_file = None
    for p in env_paths:
        if p.exists():
            env_file = p
            break
    
    if not env_file:
        print(f"⚠️ Warning: .env not found. Using env_file: vars only.")
        return False
    
    print(f"📄 Loading environment from {env_file.resolve()}...")
    loaded_count = 0
    with open(env_file, "r") as f:
        for line in f:
            line = line.strip()
            if not line or line.startswith("#"):
                continue
            if "=" in line:
                key, value = line.split("=", 1)
                key, value = key.strip(), value.strip()
                if key not in os.environ:
                    os.environ[key] = value
                    loaded_count += 1
    print(f"✅ Environment loaded: {loaded_count} new variables from {env_file.name}")
    return True

load_env_file()

# ============================================
# Step 2: Helper Functions for Testing
# ============================================
def test_tcp_connection(host, port, timeout=5, service_name="Service"):
    """Test basic TCP connectivity to a host:port"""
    try:
        sock = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
        sock.settimeout(timeout)
        result = sock.connect_ex((host, port))
        sock.close()
        if result == 0:
            print(f"✅ TCP: {service_name} reachable at {host}:{port}")
            return True
        else:
            print(f"❌ TCP: {service_name} NOT reachable at {host}:{port} (code: {result})")
            return False
    except socket.gaierror:
        print(f"❌ TCP: Cannot resolve hostname '{host}' for {service_name}")
        return False
    except Exception as e:
        print(f"❌ TCP: Error testing {service_name}: {e}")
        return False

def test_spark_jdbc_connection(spark, url, user, password, driver, table, source_name):
    """Test JDBC connection via Spark"""
    try:
        if table:
            dbtable_param = f"(SELECT * FROM {table} WHERE 1=0) as tmp"
            test_desc = f"schema from table '{table}'"
        else:
            dbtable_param = "(SELECT 1 as connection_test) as tmp"
            test_desc = "basic connectivity"
        
        df = spark.read \
            .format("jdbc") \
            .option("url", url) \
            .option("dbtable", dbtable_param) \
            .option("user", user) \
            .option("password", password) \
            .option("driver", driver) \
            .option("fetchsize", "1") \
            .load()
        
        print(f"✅ JDBC: {source_name} connection successful ({test_desc}, user: {user})")
        return True, df
    except Exception as e:
        error_msg = str(e).lower()
        if "classnotfoundexception" in error_msg:
            print(f"❌ JDBC: {source_name} driver not found - check JARs! (user: {user})")
        elif "communications link failure" in error_msg or "connection refused" in error_msg:
            print(f"❌ JDBC: {source_name} network error - check host/port/firewall (user: {user})")
        elif "authentication failed" in error_msg or "access denied" in error_msg or "scram" in error_msg:
            print(f"❌ JDBC: {source_name} authentication failed for user '{user}' - check credentials")
            if "scram" in error_msg and "empty" in error_msg:
                print(f"   💡 Hint: Password may be empty - verify .env file")
        elif "does not exist" in error_msg or "table not found" in error_msg or "relation" in error_msg:
            if table:
                print(f"⚠️ JDBC: {source_name} connected (user: {user}) but table '{table}' not found")
            else:
                print(f"❌ JDBC: {source_name} connection failed: {type(e).__name__}")
            return True if not table else False, None
        else:
            print(f"❌ JDBC: {source_name} error (user: {user}): {type(e).__name__}: {str(e)[:200]}")
        return False, None

def test_doris_connector(spark, fenodes, database, table, user, password, source_name="Doris"):
    """Test Doris connection using Spark-Doris Connector"""
    try:
        if table:
            df = spark.read \
                .format("doris") \
                .option("doris.table.identifier", f"{database}.{table}") \
                .option("doris.fenodes", fenodes) \
                .option("user", user) \
                .option("password", password) \
                .option("doris.filter.query", "1=0") \
                .load()
            print(f"✅ {source_name} Connector: Connection successful (schema from {database}.{table}, user: {user})")
            return True, df
        else:
            import urllib.request
            import json
            health_url = f"http://{fenodes}/api/health"
            try:
                with urllib.request.urlopen(health_url, timeout=10) as response:
                    health = json.loads(response.read().decode())
                    if health.get("status") == "ok":
                        print(f"✅ {source_name} Connector: FE reachable at {fenodes} (user: {user}, connection-only mode)")
                        return True, None
                    else:
                        print(f"⚠️ {source_name} FE health check returned: {health.get('status')} (user: {user})")
                        return True, None
            except Exception as http_err:
                print(f"⚠️ {source_name} HTTP health check failed, trying minimal read (user: {user})...")
                spark.read.format("doris").option("doris.fenodes", fenodes).option("user", user).option("password", password)
                print(f"✅ {source_name} Connector: JAR loaded and FE configured (user: {user}, connection-only mode)")
                return True, None
    except Exception as e:
        error_msg = str(e).lower()
        if "classnotfoundexception" in error_msg:
            print(f"❌ {source_name}: Connector JAR not loaded - check spark.jars config! (user: {user})")
        elif "stream load failed" in error_msg or "label already exists" in error_msg:
            print(f"⚠️ {source_name}: Connector loaded but write issue (user: {user}, expected for read test)")
            return True, None
        elif "connection refused" in error_msg or "timeout" in error_msg or "unreachable" in error_msg:
            print(f"❌ {source_name}: Network error to FE {fenodes} (user: {user})")
        elif "access denied" in error_msg or "authentication" in error_msg:
            print(f"❌ {source_name}: Authentication failed for user '{user}' - check credentials")
        elif "table" in error_msg and ("not exist" in error_msg or "not found" in error_msg):
            if table:
                print(f"⚠️ {source_name}: Connected (user: {user}) but table '{database}.{table}' not found")
            else:
                print(f"❌ {source_name}: Connection failed: {type(e).__name__} (user: {user})")
            return True if not table else False, None
        else:
            print(f"❌ {source_name}: Error (user: {user}): {type(e).__name__}: {str(e)[:200]}")
        return False, None

# ============================================
# Step 3: Initialize Spark Session
# ============================================
print("🚀 Initializing Spark Session...")
try:
    # ✅ FIX 1: Use comma (not colon) and correct key "spark.jars"
    # ✅ FIX 2: Include jt400-21.0.6.jar for AS/400 support
    spark_jars = "/jars/spark-doris-connector-spark-3.5-25.2.0.jar,/jars/postgresql-42.7.2.jar,/jars/jt400-21.0.6.jar"
    
    spark = SparkSession.builder \
        .appName(os.getenv("SPARK_APP_NAME", "connection_test")) \
        .master(os.getenv("SPARK_MASTER", "spark://spark-master:7077")) \
        .config("spark.jars", spark_jars) \
        .config("spark.driver.host", "jupyter-lab") \
        .config("spark.driver.bindAddress", "0.0.0.0") \
        .config("spark.executor.heartbeatInterval", "60s") \
        .config("spark.network.timeout", "800s") \
        .getOrCreate()
    
    print(f"✅ Spark Session: {spark.version} @ {spark.sparkContext.master}")
    jars_conf = spark.sparkContext.getConf().get("spark.jars", "none")
    print(f"✅ JARs loaded: {jars_conf}")
except Exception as e:
    print(f"❌ Failed to create Spark session: {e}")
    spark = None

# ============================================
# Step 4: Test PostgreSQL Connection
# ============================================
print("\n" + "="*60)
print("🐘 Testing PostgreSQL Connection")
print("="*60)

pg_host = os.getenv("POSTGRES_HOST", "host.docker.internal")
pg_port = os.getenv("POSTGRES_PORT", "5432")
pg_db = os.getenv("POSTGRES_DB", "source_db")
pg_user = os.getenv("POSTGRES_USER", "etl_user")
pg_pass = os.getenv("POSTGRES_PASSWORD", "")
pg_driver = os.getenv("POSTGRES_JDBC_DRIVER", "org.postgresql.Driver")

pg_url = f"jdbc:postgresql://{pg_host}:{pg_port}/{pg_db}"
print(f"🔗 Config: {pg_url}")
print(f"👤 User: {pg_user} | Table: {POSTGRES_TEST_TABLE or 'None (connection-only)'}")

pg_tcp_ok = test_tcp_connection(pg_host, int(pg_port), service_name="PostgreSQL")

if pg_tcp_ok and spark:
    pg_ok, pg_df = test_spark_jdbc_connection(
        spark, pg_url, pg_user, pg_pass, pg_driver, POSTGRES_TEST_TABLE, "PostgreSQL"
    )
    if pg_ok and pg_df is not None and POSTGRES_TEST_TABLE:
        print(f"📊 PostgreSQL schema preview:")
        pg_df.printSchema()
else:
    pg_ok = False

# ============================================
# Step 5: Test Apache Doris Connection
# ============================================
print("\n" + "="*60)
print("🔥 Testing Apache Doris Connection")
print("="*60)

doris_fenodes = f"{os.getenv('DORIS_FE_HOST', 'host.docker.internal')}:{os.getenv('DORIS_FE_HTTP_PORT', '8030')}"
doris_db = os.getenv("DORIS_DB", "target_db")
doris_user = os.getenv("DORIS_USER", "root")
doris_pass = os.getenv("DORIS_PASSWORD", "")

print(f"🔗 Config: FE={doris_fenodes} | DB={doris_db}")
print(f"👤 User: {doris_user} | Table: {DORIS_TEST_TABLE or 'None (connection-only)'}")

doris_tcp_ok = test_tcp_connection(
    os.getenv("DORIS_FE_HOST", "host.docker.internal"), 
    int(os.getenv("DORIS_FE_HTTP_PORT", "8030")), 
    service_name="Doris FE (HTTP)"
)

if doris_tcp_ok and spark:
    doris_ok, doris_df = test_doris_connector(
        spark, doris_fenodes, doris_db, DORIS_TEST_TABLE, doris_user, doris_pass, "Doris"
    )
    if doris_ok and doris_df is not None and DORIS_TEST_TABLE:
        print(f"📊 Doris schema preview:")
        doris_df.printSchema()
else:
    doris_ok = False

# ============================================
# 🔍 DEBUG: Check DB2I_ENABLED Value
# ============================================
print("\n🔍 DEBUG: DB2I_ENABLED Check")
print("-" * 50)
db2i_raw = os.getenv("DB2I_ENABLED", "NOT SET")
print(f"Raw value: '{db2i_raw}'")
print(f"Lowercase: '{db2i_raw.lower()}'")
print(f"Equals 'true': {db2i_raw.lower() == 'true'}")
print(f"Length: {len(db2i_raw)}")
print(f"Repr: {repr(db2i_raw)}")

# ============================================
# Step 6: Test IBM DB2i Connection (Optional)
# ============================================
print("\n" + "="*60)
print("🔷 Testing IBM DB2i Connection (Optional)")
print("="*60)

db2i_enabled = os.getenv("DB2I_ENABLED", "false").lower() == "true"

if not db2i_enabled:
    print("⏭️  DB2i testing disabled (set DB2I_ENABLED=true in .env to enable)")
    db2i_ok = None
elif not spark:
    print("⚠️  Skipping DB2i test: Spark session not available")
    db2i_ok = False
else:
    db2i_host = os.getenv("DB2I_HOST", "")
    
    # ✅ FIX 3: Safe port parsing - handle empty string
    db2i_port_raw = os.getenv("DB2I_PORT", "446").strip()
    db2i_port = int(db2i_port_raw) if db2i_port_raw else 446
    
    db2i_db = os.getenv("DB2I_DB", "")
    db2i_schema = os.getenv("DB2I_SCHEMA", "")
    db2i_user = os.getenv("DB2I_USER", "")
    db2i_pass = os.getenv("DB2I_PASSWORD", "")
    
    # ✅ FIX 4: Use AS/400 driver (not standard DB2 driver)
    db2i_driver = os.getenv("DB2I_JDBC_DRIVER", "com.ibm.as400.access.AS400JDBCDriver")
    
    # ✅ FIX 5: Use AS/400 URL template (jdbc:as400:// not jdbc:db2://)
    db2i_template = os.getenv("DB2I_JDBC_URL_TEMPLATE", "jdbc:as400://{host};libraries={schema};")
    
    # Build JDBC URL
    db2i_url = db2i_template.format(
        host=db2i_host,
        port=db2i_port,
        database=db2i_db,
        schema=db2i_schema
    )
    
    print(f"🔗 Config: {db2i_url.replace(db2i_pass, '***')}")
    print(f"👤 User: {db2i_user} | Table: {DB2I_TEST_TABLE or 'None (connection-only)'} | Port: {db2i_port}")
    
    # Test TCP first
    db2i_tcp_ok = test_tcp_connection(db2i_host, db2i_port, service_name="DB2i")
    
    if db2i_tcp_ok:
        db2i_ok, db2i_df = test_spark_jdbc_connection(
            spark, db2i_url, db2i_user, db2i_pass, db2i_driver, DB2I_TEST_TABLE, "DB2i"
        )
        if db2i_ok and db2i_df is not None and DB2I_TEST_TABLE:
            print(f"📊 DB2i schema preview:")
            db2i_df.printSchema()
    else:
        db2i_ok = False

# ============================================
# Step 7: Summary Report
# ============================================
print("\n" + "="*60)
print("📋 CONNECTION TEST SUMMARY")
print("="*60)

results = {
    "PostgreSQL": pg_ok,
    "Apache Doris": doris_ok,
    "IBM DB2i": db2i_ok if db2i_enabled else "⏭️ Disabled"
}

for service, status in results.items():
    if status is True:
        icon = "✅"
    elif status is False:
        icon = "❌"
    else:
        icon = "⏭️"
    print(f"{icon} {service}: {status}")

all_ready = pg_ok and doris_ok and (db2i_ok is not False if db2i_enabled else True)
if all_ready and spark:
    print(f"\n🎉 All enabled connections successful! Ready for ETL pipeline.")
elif spark:
    print(f"\n⚠️ Some connections failed. Check errors above before proceeding.")
else:
    print(f"\n❌ Spark session failed to initialize. Fix Spark config first.")

# ============================================
# Step 8: Debug Info
# ============================================
print("\n🔍 Debug: Environment & Config Verification")
print("-" * 50)
print(f"📁 .env readable: {Path('/home/jovyan/.env').exists()}")
print(f"👤 PostgreSQL user: {os.getenv('POSTGRES_USER', 'NOT SET')}")
print(f"👤 Doris user: {os.getenv('DORIS_USER', 'NOT SET')}")
print(f"👤 DB2i user: {os.getenv('DB2I_USER', 'NOT SET')}")
print(f"🔐 POSTGRES_PASSWORD set: {'✅ Yes' if os.getenv('POSTGRES_PASSWORD') else '❌ No'}")
print(f"📦 JARs config: {spark_jars if 'spark_jars' in locals() else 'N/A'}")

# ============================================
# Step 9: Cleanup (Optional)
# ============================================
# spark.stop()
# print("✨ Spark session stopped")

In [ ]:
# 🔍 DB2i Port Diagnostic

# %%
import socket
import os

db2i_host = os.getenv("DB2I_HOST", "")
ports_to_try = [446, 50000, 8471, 23]  # Common AS/400 ports

print(f"🔍 Testing connectivity to {db2i_host} from Docker container")
print("-" * 70)

for port in ports_to_try:
    try:
        sock = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
        sock.settimeout(5)
        result = sock.connect_ex((db2i_host, port))
        sock.close()
        
        status = "✅ OPEN" if result == 0 else f"❌ Closed (code: {result})"
        print(f"Port {port:5d}: {status}")
    except socket.gaierror:
        print(f"Port {port:5d}: ❌ Cannot resolve hostname '{db2i_host}'")
        break
    except Exception as e:
        print(f"Port {port:5d}: ❌ Error: {e}")

# Test hostname resolution
try:
    ip = socket.gethostbyname(db2i_host)
    print(f"\n✅ Hostname resolved: {db2i_host} → {ip}")
except socket.gaierror:
    print(f"\n❌ Cannot resolve hostname: {db2i_host}")
    print("💡 Fix: Add to docker-compose.yml extra_hosts:")
    print(f"   extra_hosts:")
    print(f"     - \"{db2i_host}:<IP_ADDRESS>\"")

In [2]:
# 🔗 Connection Test: PostgreSQL + Doris + DB2i 222
# Tests connectivity to all data sources using environment variables from mounted .env
# ✨ Features: Optional table check + Username logging + jt400 SSL options + Better error reporting

# %%
import os
import sys
from pathlib import Path
from pyspark.sql import SparkSession
from pyspark.sql.functions import lit, current_timestamp
import socket
import json
import urllib.request

# ============================================
# 🔧 CONFIGURATION: Optional Table Names
# Set to None or "" to skip table-specific checks (connection-only mode)
# ============================================
POSTGRES_TEST_TABLE = os.getenv("POSTGRES_TEST_TABLE", "").strip() or None
DORIS_TEST_TABLE = os.getenv("DORIS_TEST_TABLE", "").strip() or None
DB2I_TEST_TABLE = os.getenv("DB2I_TEST_TABLE", "").strip() or None

print(f"🔧 Config: POSTGRES_TEST_TABLE={POSTGRES_TEST_TABLE}, DORIS_TEST_TABLE={DORIS_TEST_TABLE}, DB2I_TEST_TABLE={DB2I_TEST_TABLE}")

# ============================================
# Step 1: Load Environment Variables from Mounted .env
# ============================================
def load_env_file(env_path="/home/jovyan/.env"):
    """Load .env file into os.environ with fallback paths"""
    env_paths = [
        Path(env_path),
        Path("/home/jovyan/.env"),
        Path(".env"),
        Path("/home/jovyan/work/.env")
    ]
    
    env_file = None
    for p in env_paths:
        if p.exists():
            env_file = p
            break
    
    if not env_file:
        print(f"⚠️ Warning: .env not found. Using env_file: vars only.")
        return False
    
    print(f"📄 Loading environment from {env_file.resolve()}...")
    loaded_count = 0
    with open(env_file, "r") as f:
        for line in f:
            line = line.strip()
            if not line or line.startswith("#"):
                continue
            if "=" in line:
                key, value = line.split("=", 1)
                key, value = key.strip(), value.strip()
                if key not in os.environ:
                    os.environ[key] = value
                    loaded_count += 1
    print(f"✅ Environment loaded: {loaded_count} new variables from {env_file.name}")
    return True

load_env_file()

# ============================================
# Step 2: Helper Functions for Testing
# ============================================
def test_tcp_connection(host, port, timeout=5, service_name="Service"):
    """Test basic TCP connectivity to a host:port"""
    try:
        sock = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
        sock.settimeout(timeout)
        result = sock.connect_ex((host, port))
        sock.close()
        if result == 0:
            print(f"✅ TCP: {service_name} reachable at {host}:{port}")
            return True
        else:
            print(f"❌ TCP: {service_name} NOT reachable at {host}:{port} (code: {result})")
            return False
    except socket.gaierror:
        print(f"❌ TCP: Cannot resolve hostname '{host}' for {service_name}")
        return False
    except Exception as e:
        print(f"❌ TCP: Error testing {service_name}: {e}")
        return False

def test_spark_jdbc_connection(spark, url, user, password, driver, table, source_name, extra_options=None):
    """
    Test JDBC connection via Spark.
    
    KEY FIX: DB2i (AS/400) does not support SELECT without FROM clause.
    Every DB2i test query must use SYSIBM.SYSDUMMY1 as the dummy table.
    This is equivalent to Oracle's DUAL table.
    
    Supported databases and their dummy table:
      DB2i / AS400  → SYSIBM.SYSDUMMY1   (built-in, always exists)
      Oracle        → DUAL
      PostgreSQL    → no dummy needed     (SELECT 1 works)
      MySQL         → no dummy needed     (SELECT 1 works)
    """
    try:
        if table:
            # Table-specific: read schema only (WHERE 1=0 returns no rows)
            if source_name == "DB2i":
                # DB2i needs qualified schema — table must be SCHEMA.TABLE or just TABLE
                dbtable_param = f"(SELECT * FROM {table} WHERE 1=0) as tmp"
            else:
                dbtable_param = f"(SELECT * FROM {table} WHERE 1=0) as tmp"
            test_desc = f"schema from table '{table}'"
        else:
            # Connection-only: just ping the database with a no-op query
            if source_name == "DB2i":
                # ✅ FIXED: DB2i requires FROM clause — use SYSIBM.SYSDUMMY1
                # This table always exists on every IBM i / AS400 system
                dbtable_param = "(SELECT 1 FROM SYSIBM.SYSDUMMY1 WHERE 1=0) as tmp"
            else:
                # PostgreSQL, MySQL, etc. — bare SELECT 1 is fine
                dbtable_param = "(SELECT 1 as connection_test) as tmp"
            test_desc = "basic connectivity"

        reader = (
            spark.read
            .format("jdbc")
            .option("url", url)
            .option("dbtable", dbtable_param)
            .option("user", user)
            .option("password", password)
            .option("driver", driver)
            .option("fetchsize", "1")
        )

        # Apply extra options (e.g. jt400 properties)
        if extra_options:
            for key, value in extra_options.items():
                reader = reader.option(key, value)

        df = reader.load()
        print(f"✅ JDBC: {source_name} connection successful ({test_desc})")
        return True, df

    except Exception as e:
        java_exception = getattr(e, 'java_exception', None)
        if java_exception:
            try:
                java_msg = java_exception.toString() if hasattr(java_exception, 'toString') else str(java_exception)
                print(f"❌ JDBC: {source_name} Java exception: {java_msg}")
            except:
                pass

        error_msg = str(e).lower()
        if "sql0104" in error_msg or "token" in error_msg and "valid" in error_msg:
            print(f"❌ JDBC: {source_name} SQL SYNTAX error — DB2i needs FROM clause in every SELECT")
            print(f"   Fix: Use SYSIBM.SYSDUMMY1 as dummy table for connection tests")
        elif "classnotfoundexception" in error_msg:
            print(f"❌ JDBC: {source_name} driver JAR not found — check ./jars/ folder")
        elif "connection refused" in error_msg or "communications" in error_msg:
            print(f"❌ JDBC: {source_name} network unreachable — check host/port/firewall")
        elif "as400securityexception" in error_msg or "authentication" in error_msg:
            print(f"❌ JDBC: {source_name} authentication failed — check user/password")
        elif "as400exception" in error_msg:
            print(f"❌ JDBC: {source_name} AS400 error — check library name and permissions")
        else:
            print(f"❌ JDBC: {source_name} failed: {str(e)[:400]}")

        return False, None

def test_doris_connector(spark, fenodes, database, table, user, password, source_name="Doris"):
    """Test Doris connection using Spark-Doris Connector"""
    try:
        if table:
            df = spark.read \
                .format("doris") \
                .option("doris.table.identifier", f"{database}.{table}") \
                .option("doris.fenodes", fenodes) \
                .option("user", user) \
                .option("password", password) \
                .option("doris.filter.query", "1=0") \
                .load()
            print(f"✅ {source_name} Connector: Connection successful (schema from {database}.{table}, user: {user})")
            return True, df
        else:
            # Connection-only test: try to list databases or use a minimal query
            health_url = f"http://{fenodes}/api/health"
            try:
                with urllib.request.urlopen(health_url, timeout=10) as response:
                    health = json.loads(response.read().decode())
                    if health.get("status") == "ok":
                        print(f"✅ {source_name} Connector: FE reachable at {fenodes} (user: {user}, connection-only mode)")
                        return True, None
                    else:
                        print(f"⚠️ {source_name} FE health check returned: {health.get('status')} (user: {user})")
                        return True, None
            except Exception as http_err:
                print(f"⚠️ {source_name} HTTP health check failed, trying minimal read (user: {user})...")
                spark.read.format("doris").option("doris.fenodes", fenodes).option("user", user).option("password", password)
                print(f"✅ {source_name} Connector: JAR loaded and FE configured (user: {user}, connection-only mode)")
                return True, None
    except Exception as e:
        error_msg = str(e).lower()
        if "classnotfoundexception" in error_msg:
            print(f"❌ {source_name}: Connector JAR not loaded - check spark.jars config! (user: {user})")
        elif "stream load failed" in error_msg or "label already exists" in error_msg:
            print(f"⚠️ {source_name}: Connector loaded but write issue (user: {user}, expected for read test)")
            return True, None
        elif "connection refused" in error_msg or "timeout" in error_msg or "unreachable" in error_msg:
            print(f"❌ {source_name}: Network error to FE {fenodes} (user: {user})")
        elif "access denied" in error_msg or "authentication" in error_msg:
            print(f"❌ {source_name}: Authentication failed for user '{user}' - check credentials")
        elif "table" in error_msg and ("not exist" in error_msg or "not found" in error_msg):
            if table:
                print(f"⚠️ {source_name}: Connected (user: {user}) but table '{database}.{table}' not found")
            else:
                print(f"❌ {source_name}: Connection failed: {type(e).__name__} (user: {user})")
            return True if not table else False, None
        else:
            print(f"❌ {source_name}: Error (user: {user}): {type(e).__name__}: {str(e)[:200]}")
        return False, None

# ============================================
# Step 3: Initialize Spark Session
# ============================================
print("🚀 Initializing Spark Session...")
try:
    # ✅ Include jt400-21.0.6.jar for AS/400 support
    spark_jars = "/jars/spark-doris-connector-spark-3.5-25.2.0.jar,/jars/postgresql-42.7.2.jar,/jars/jt400-21.0.6.jar"
    
    spark = SparkSession.builder \
        .appName(os.getenv("SPARK_APP_NAME", "connection_test")) \
        .master(os.getenv("SPARK_MASTER", "spark://spark-master:7077")) \
        .config("spark.jars", spark_jars) \
        .config("spark.driver.host", "jupyter-lab") \
        .config("spark.driver.bindAddress", "0.0.0.0") \
        .config("spark.executor.heartbeatInterval", "60s") \
        .config("spark.network.timeout", "800s") \
        .config("spark.driver.extraJavaOptions", "-Dcom.ibm.as400.access.Trace=4 -Dcom.ibm.as400.access.TraceFileName=/tmp/jt400.log") \
        .config("spark.executor.extraJavaOptions", "-Dcom.ibm.as400.access.Trace=4 -Dcom.ibm.as400.access.TraceFileName=/tmp/jt400.log") \
        .getOrCreate()
    
    print(f"✅ Spark Session: {spark.version} @ {spark.sparkContext.master}")
    jars_conf = spark.sparkContext.getConf().get("spark.jars", "none")
    print(f"✅ JARs loaded: {jars_conf}")
except Exception as e:
    print(f"❌ Failed to create Spark session: {e}")
    spark = None

# ============================================
# Step 4: Test PostgreSQL Connection
# ============================================
print("\n" + "="*60)
print("🐘 Testing PostgreSQL Connection")
print("="*60)

pg_host = os.getenv("POSTGRES_HOST", "host.docker.internal")
pg_port = os.getenv("POSTGRES_PORT", "5432")
pg_db = os.getenv("POSTGRES_DB", "source_db")
pg_user = os.getenv("POSTGRES_USER", "etl_user")
pg_pass = os.getenv("POSTGRES_PASSWORD", "")
pg_driver = os.getenv("POSTGRES_JDBC_DRIVER", "org.postgresql.Driver")

pg_url = f"jdbc:postgresql://{pg_host}:{pg_port}/{pg_db}"
print(f"🔗 Config: {pg_url}")
print(f"👤 User: {pg_user} | Table: {POSTGRES_TEST_TABLE or 'None (connection-only)'}")

pg_tcp_ok = test_tcp_connection(pg_host, int(pg_port), service_name="PostgreSQL")

if pg_tcp_ok and spark:
    pg_ok, pg_df = test_spark_jdbc_connection(
        spark, pg_url, pg_user, pg_pass, pg_driver, POSTGRES_TEST_TABLE, "PostgreSQL"
    )
    if pg_ok and pg_df is not None and POSTGRES_TEST_TABLE:
        print(f"📊 PostgreSQL schema preview:")
        pg_df.printSchema()
else:
    pg_ok = False

# ============================================
# Step 5: Test Apache Doris Connection
# ============================================
print("\n" + "="*60)
print("🔥 Testing Apache Doris Connection")
print("="*60)

doris_fenodes = f"{os.getenv('DORIS_FE_HOST', 'host.docker.internal')}:{os.getenv('DORIS_FE_HTTP_PORT', '8030')}"
doris_db = os.getenv("DORIS_DB", "target_db")
doris_user = os.getenv("DORIS_USER", "root")
doris_pass = os.getenv("DORIS_PASSWORD", "")

print(f"🔗 Config: FE={doris_fenodes} | DB={doris_db}")
print(f"👤 User: {doris_user} | Table: {DORIS_TEST_TABLE or 'None (connection-only)'}")

doris_tcp_ok = test_tcp_connection(
    os.getenv("DORIS_FE_HOST", "host.docker.internal"), 
    int(os.getenv("DORIS_FE_HTTP_PORT", "8030")), 
    service_name="Doris FE (HTTP)"
)

if doris_tcp_ok and spark:
    doris_ok, doris_df = test_doris_connector(
        spark, doris_fenodes, doris_db, DORIS_TEST_TABLE, doris_user, doris_pass, "Doris"
    )
    if doris_ok and doris_df is not None and DORIS_TEST_TABLE:
        print(f"📊 Doris schema preview:")
        doris_df.printSchema()
else:
    doris_ok = False

# ============================================
# 🔍 DEBUG: Check DB2I_ENABLED Value
# ============================================
print("\n🔍 DEBUG: DB2I_ENABLED Check")
print("-" * 50)
db2i_raw = os.getenv("DB2I_ENABLED", "NOT SET")
print(f"Raw value: '{db2i_raw}'")
print(f"Lowercase: '{db2i_raw.lower()}'")
print(f"Equals 'true': {db2i_raw.lower() == 'true'}")
print(f"Length: {len(db2i_raw)}")
print(f"Repr: {repr(db2i_raw)}")

# ============================================
# Step 6: Test IBM DB2i Connection (Optional)
# ============================================
print("\n" + "="*60)
print("🔷 Testing IBM DB2i Connection (Optional)")
print("="*60)

db2i_enabled = os.getenv("DB2I_ENABLED", "false").lower() == "true"

if not db2i_enabled:
    print("⏭️  DB2i testing disabled (set DB2I_ENABLED=true in .env to enable)")
    db2i_ok = None
elif not spark:
    print("⚠️  Skipping DB2i test: Spark session not available")
    db2i_ok = False
else:
    db2i_host = os.getenv("DB2I_HOST", "")
    
    # ✅ Safe port parsing - handle empty string (default to 8471 for IBM i Access)
    db2i_port_raw = os.getenv("DB2I_PORT", "8471").strip()
    db2i_port = int(db2i_port_raw) if db2i_port_raw else 8471
    
    db2i_db = os.getenv("DB2I_DB", "")
    db2i_schema = os.getenv("DB2I_SCHEMA", "").strip().upper()  # AS/400 libraries are case-sensitive
    db2i_user = os.getenv("DB2I_USER", "")
    db2i_pass = os.getenv("DB2I_PASSWORD", "")
    
    # ✅ Use AS/400 driver (not standard DB2 driver)
    db2i_driver = os.getenv("DB2I_JDBC_DRIVER", "com.ibm.as400.access.AS400JDBCDriver")
    
    # ✅ Build URL WITHOUT inline properties (properties go as .option() calls)
    db2i_url = f"jdbc:as400://{db2i_host}:{db2i_port};libraries={db2i_schema};"
    
    # ✅ jt400-specific options as .option() calls (NOT in URL)
    db2i_extra_options = {
    "secure": "false",          # SSL/TLS — boolean string, not integer
    "timeout": "30",           # Connection timeout in seconds
    "socketTimeout": "60",     # Socket read timeout in seconds
    "prompt": "false",         # Disable the AS/400 sign-on GUI popup dialog
    "naming": "sql",           # Use SQL naming convention (schema.table) not system (schema/table)
    "translate binary": "true",# Handle binary data correctly
    }
    
    print(f"🔗 Config: {db2i_url.replace(db2i_pass, '***')}")
    print(f"👤 User: {db2i_user} | Table: {DB2I_TEST_TABLE or 'None (connection-only)'} | Port: {db2i_port}")
    print(f"⚙️  jt400 options: {db2i_extra_options}")
    
    # Test TCP first
    db2i_tcp_ok = test_tcp_connection(db2i_host, db2i_port, service_name="DB2i")
    
    if db2i_tcp_ok:
        # ✅ Pass extra_options to test_spark_jdbc_connection
        db2i_ok, db2i_df = test_spark_jdbc_connection(
            spark, 
            db2i_url, 
            db2i_user, 
            db2i_pass, 
            db2i_driver, 
            DB2I_TEST_TABLE, 
            "DB2i",
            extra_options=db2i_extra_options
        )
        if db2i_ok and db2i_df is not None and DB2I_TEST_TABLE:
            print(f"📊 DB2i schema preview:")
            db2i_df.printSchema()
    else:
        db2i_ok = False

# ============================================
# Step 7: Summary Report
# ============================================
print("\n" + "="*60)
print("📋 CONNECTION TEST SUMMARY")
print("="*60)

results = {
    "PostgreSQL": pg_ok,
    "Apache Doris": doris_ok,
    "IBM DB2i": db2i_ok if db2i_enabled else "⏭️ Disabled"
}

for service, status in results.items():
    if status is True:
        icon = "✅"
    elif status is False:
        icon = "❌"
    else:
        icon = "⏭️"
    print(f"{icon} {service}: {status}")

all_ready = pg_ok and doris_ok and (db2i_ok is not False if db2i_enabled else True)
if all_ready and spark:
    print(f"\n🎉 All enabled connections successful! Ready for ETL pipeline.")
elif spark:
    print(f"\n⚠️ Some connections failed. Check errors above before proceeding.")
else:
    print(f"\n❌ Spark session failed to initialize. Fix Spark config first.")

# ============================================
# Step 8: Debug Info
# ============================================
print("\n🔍 Debug: Environment & Config Verification")
print("-" * 50)
print(f"📁 .env readable: {Path('/home/jovyan/.env').exists()}")
print(f"👤 PostgreSQL user: {os.getenv('POSTGRES_USER', 'NOT SET')}")
print(f"👤 Doris user: {os.getenv('DORIS_USER', 'NOT SET')}")
print(f"👤 DB2i user: {os.getenv('DB2I_USER', 'NOT SET')}")
print(f"🔐 POSTGRES_PASSWORD set: {'✅ Yes' if os.getenv('POSTGRES_PASSWORD') else '❌ No'}")
print(f"📦 JARs config: {spark_jars if 'spark_jars' in locals() else 'N/A'}")

# 🔍 Show jt400 trace log location if enabled
print(f"📝 jt400 trace log: /tmp/jt400.log (check inside container with: docker exec jupyter-lab cat /tmp/jt400.log)")

# ============================================
# Step 9: Cleanup (Optional)
# ============================================
# spark.stop()
# print("✨ Spark session stopped")

🔧 Config: POSTGRES_TEST_TABLE=None, DORIS_TEST_TABLE=None, DB2I_TEST_TABLE=None
📄 Loading environment from /home/jovyan/.env...
✅ Environment loaded: 0 new variables from .env
🚀 Initializing Spark Session...
✅ Spark Session: 3.5.0 @ spark://spark-master:7077
✅ JARs loaded: /jars/spark-doris-connector-spark-3.5-25.2.0.jar,/jars/postgresql-42.7.2.jar,/jars/jt400-21.0.6.jar

🐘 Testing PostgreSQL Connection
🔗 Config: jdbc:postgresql://host.docker.internal:5432/learning_db
👤 User: learner | Table: None (connection-only)
✅ TCP: PostgreSQL reachable at host.docker.internal:5432
✅ JDBC: PostgreSQL connection successful (basic connectivity)

🔥 Testing Apache Doris Connection
🔗 Config: FE=host.docker.internal:8030 | DB=target_db
👤 User: root | Table: None (connection-only)
✅ TCP: Doris FE (HTTP) reachable at host.docker.internal:8030
⚠️ Doris FE health check returned: None (user: root)

🔍 DEBUG: DB2I_ENABLED Check
--------------------------------------------------
Raw value: 'true'
Lowercase: 't

In [ ]:
# Debug: Show what options were actually used
if db2i_enabled and db2i_tcp_ok:
    print("\n🔍 Debug: jt400 Options Applied")
    print("-" * 50)
    for key, value in db2i_extra_options.items():
        print(f"   .option(\"{key}\", \"{value}\")")

In [ ]:
# 🔍 Direct jt400 Test (No Spark)

# %%
import subprocess
import sys

# Test if jt400 JAR is loadable by Java
result = subprocess.run([
    "java", "-cp", "/jars/jt400-21.0.6.jar",
    "com.ibm.as400.access.AS400JDBCDriver"
], capture_output=True, text=True, timeout=10)

if result.returncode == 0 or "main method" in result.stderr:
    print("✅ jt400 driver class loads successfully")
else:
    print(f"❌ jt400 driver load failed: {result.stderr[:200]}")

# Test basic JDBC URL parsing (won't connect, just validates format)
try:
    from java.sql import DriverManager  # Requires jpype
    print("✅ Java SQL classes accessible")
except ImportError:
    print("⚠️ jpype not available - skip direct Java test")